# Lab 5: Deep Learning

This lab consists of three parts that together form a complete pipeline for applying a deep neural network to a multi class classification task.

- Part 1: Read the provided code carefully.
- Parts 2 and 3: Use an AI model such as a large language model (e.g., GPT, Gemini, Claude) to complete the code.
- Part 4: Write a self reflection on your experience using AI in this lab.  

**Submission Requirement:**
- Submit this file only with all the running results and completed written parts.

## Part 0: Environment Setup and Data Download

In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

In [3]:
## download dataset
if not os.path.exists('Celebrity_Faces_Dataset'):
    remote_path = 'https://github.com/Michigan-State-University-CSE-440/logistic-regression/releases/download/v1.0/Celebrity_Faces_Dataset.zip'
    local_path = 'Celebrity_Faces_Dataset.zip'
    if not os.path.exists(local_path):
        os.system(f'wget {remote_path}')
    os.system(f'unzip {local_path}')

## Part 1: Data Loading

In [4]:
# ---------------------------------------------
# 1) Data Loading for the Celebrity Faces
# ---------------------------------------------
def load_celebrity_faces(root_dir, image_size=(64, 64)):
    """
    Loads images from subfolders of `root_dir` (one folder per celebrity).
    Returns:
        X_gray: Flattened grayscale images for training, shape (num_samples, H*W).
        y:      Label array, shape (num_samples,).
        label_map: {label_idx: 'celebrity_name'}.
        X_color:  Color images (resized but not converted to grayscale),
                  shape (num_samples, H, W, 3) in RGB order for display.
    """
    X_input = []
    X_color = []
    y = []

    # subfolders = celebrity names
    classes = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])

    label_map = {idx: celeb for idx, celeb in enumerate(classes)}
    name_to_label = {celeb: idx for idx, celeb in enumerate(classes)}

    for celeb_name in classes:
        celeb_label = name_to_label[celeb_name]
        celeb_folder = os.path.join(root_dir, celeb_name)

        for i, filename in enumerate(os.listdir(celeb_folder)):
            if i >= 100:  # limit to 100 images per class
                break
            filepath = os.path.join(celeb_folder, filename)
            img_bgr = cv2.imread(filepath)
            if img_bgr is None:
                continue

            # Resize color image for display
            img_bgr_vis = cv2.resize(img_bgr, (256,256))
            # Convert BGR (OpenCV) to RGB for matplotlib
            img_rgb_vis = cv2.cvtColor(img_bgr_vis, cv2.COLOR_BGR2RGB)
            X_color.append(img_rgb_vis)

            # Also create a grayscale copy for training
            img_bgr = cv2.resize(img_bgr, image_size)
            X_input.append(img_bgr)

            y.append(celeb_label)

    X_color = np.array(X_color, dtype=np.uint8)  # (num_samples, H, W, 3)
    X_input = np.array(X_input, dtype=np.float32)  # (num_samples, H', W', 3)
    y = np.array(y, dtype=np.int32)

    return X_input, y, label_map, X_color


In [5]:
# ---------------------------------------------------
# 3) Label-Wise k-Fold Splitting (Manual Stratification)
# ---------------------------------------------------
def labelwise_kfold_split(X, y, k=5, shuffle=True, random_state=None):
    """
    Manually perform a label-wise k-fold split.
    Returns list of (train_indices, test_indices) for each fold.
    Ensures each label's samples appear in all folds.
    """
    assert len(X) == len(y), "X and y must have same length."

    label_indices_map = {}
    unique_labels = np.unique(y)

    # gather indices by label
    for label in unique_labels:
        label_indices = np.where(y == label)[0]
        label_indices_map[label] = label_indices

    # shuffle if needed
    rng = np.random.default_rng(random_state)
    if shuffle:
        for label in unique_labels:
            rng.shuffle(label_indices_map[label])

    # partition each label's indices into k folds
    label_folds = {}
    for label in unique_labels:
        indices = label_indices_map[label]
        num_samples_label = len(indices)

        fold_sizes = [num_samples_label // k] * k
        for i in range(num_samples_label % k):
            fold_sizes[i] += 1

        start = 0
        label_folds[label] = []
        for fold_size in fold_sizes:
            end = start + fold_size
            fold_subset = indices[start:end]
            label_folds[label].append(fold_subset)
            start = end

    # combine folds across labels
    folds = []
    for i in range(k):
        test_indices_list = []
        train_indices_list = []
        for label in unique_labels:
            # i-th subset for label -> test
            test_indices_list.append(label_folds[label][i])
            # other subsets -> train
            for j in range(k):
                if j != i:
                    train_indices_list.append(label_folds[label][j])

        test_indices = np.concatenate(test_indices_list)
        train_indices = np.concatenate(train_indices_list)
        folds.append((train_indices, test_indices))

    return folds

## Part 2: Define Framework

### Task Goal
Build a CNN-based framework with two layers:
1. `SimpleCNN` defines the neural network architecture (conv blocks + classifier).
2. `CNNClassifier` wraps model setup, training for one epoch, and validation using mini-batches.


### LLM Coding Exercise (Part 2: Framework Definition)
Use an LLM to generate the code for the masked framework cells.

Try prompting in two rounds:

1. **Direct prompt (baseline):** paste only the Task Goal.
2. **Optimized prompt:** add constraints such as:
   - Keep exact class names: `SimpleCNN`, `CNNClassifier`.
   - Input shape and channel-order expectations.
   - Use `torch`, `nn`, `optim`, `TensorDataset`, `DataLoader`.
   - `train_epoch` must retuen `(loss)`.
   - `validate` must return `(accuracy, loss)`.

Compare the two outputs and keep the more reliable version.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, input_channels=1, num_classes=10, image_size=(28, 28)):
        super(SimpleCNN, self).__init__()

        # Convolutional feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # halves H, W

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Dynamically compute flattened size
        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, *image_size)
            out = self.features(dummy)
            self.flatten_dim = out.view(1, -1).shape[1]

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(self.flatten_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


In [ ]:
class CNNClassifier:
    def __init__(self, learning_rate=0.001, num_epochs=10, batch_size=64, seed=42,
                 input_channels=1, image_size=(28, 28), num_classes=10):
        # TODO: initialize model/device/optimizer/loss

    def train_epoch(self, X, y):
        # TODO: implement one epoch of mini-batch training
        return loss

    def validate(self, X, y):
        # TODO: implement evaluation and return (accuracy, loss)
        return accuracy, loss


In [ ]:
root_dir = "Celebrity_Faces_Dataset"
learning_rate=0.0001
k=5
num_epochs=10
batch_size=32
seed=440

## Part 3: Training & Evaluating Network

### Task Goal
Build the end-to-end training loop to:
1. Load and normalize data.
2. Create manual label-wise k-fold splits.
3. Train `CNNClassifier` on each fold across epochs.
4. Validate every epoch and report fold and average accuracy.


### LLM Coding Exercise (Part 3: Training & Evaluation)
Use an LLM to generate the code for the masked training cell.

Try prompting in two rounds:

1. **Direct prompt (baseline):** paste only the Task Goal.
2. **Optimized prompt:** add constraints such as:
   - Preserve existing helpers: `load_celebrity_faces`, `labelwise_kfold_split`, `CNNClassifier`.
   - Normalize input to `[0,1]`.
   - Train by fold and by epoch; evaluate each epoch.
   - Print fold progress and final average k-fold accuracy.

Then test and refine prompts if output has shape bugs, API mismatch, or missing logs.


In [ ]:
# Suggested structure:
# 1) Load data
# 2) Normalize features
# 3) Create label-wise k-fold splits
# 4) For each fold: train and validate for multiple epochs
# 5) Report average accuracy across folds


## Part 4: Self-Reflection: AI for Coding

Answer the following questions after completing the lab:

a. Is AI sufficient for coding in this task? Explain why or why not.

b. Based on your answer to (a):  
   - If yes, in which aspects does AI perform well, or even outperform human programmers?  
   - If not, in which aspects does AI fall short, and how can human input improve the outcome?

c. How do you evaluate whether the AI-generated output is insufficient or incorrect?

d. When the AI output is not satisfactory, what specific steps do you take to improve it (e.g., prompt refinement, adding constraints, testing, debugging)?

e. How will this lab help you use AI more effectively for programming in the future? Additionally, how should one collaborate with AI to maximize both efficiency and code quality?
